<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">📋</div><div style="color:white;font-size:1.5rem;font-weight:700;">Frequency Analyser</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Word and n-gram frequency lists, type-token ratio, and Zipf distribution<br><a href="https://ladal.edu.au/tools.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2705; How to use this tool:</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;line-height:2.0;"><li>Upload your <code>.txt</code> files to <b>MyTexts</b> (Step 2). Demo texts are loaded automatically if the folder is empty.</li><li>Edit settings (n-gram size, stopwords, etc.) in the configuration cell (Step 2).</li><li>Click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the analysis.</li><li>Download results from <b>MyOutput</b> (Step 4).</li></ol></div>


<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#7a5000;">&#x26A1; Quick start:</b> Upload your files to the appropriate folder, edit the configuration cell, then click <b>Kernel &#x2192; Restart &amp; Run All</b>. The notebook runs from top to bottom automatically.</div>


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 1</span><br><b style="font-size:.98rem;">&#x2699;&#xFE0F; Set up the environment</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Step 1 — Load shared helpers (do not edit)
source("../helpers/ladal_helpers.R")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 2</span><br><b style="font-size:.98rem;">&#x1F4C2; Upload your texts &amp; configure</b></div>


<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files to <code>MyTexts</code></b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Open the <b>file browser panel</b> on the left (click the folder icon if hidden).</li><li>Double-click the <b><code>MyTexts</code></b> folder.</li><li><b>Drag and drop</b> your <code>.txt</code> files into that folder.</li><li>Then click <b>Kernel &#x2192; Restart &amp; Run All</b>.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.txt</code> files are accepted. If the folder is empty, demo data will be loaded automatically.</p></div>


<div style="background:#e8f4fd;border-left:4px solid #4085C6;border-radius:4px;padding:7px 14px;margin-bottom:3px;font-family:sans-serif;font-size:.82rem;color:#2a5ea8;">&#x270F;&#xFE0F; <b>Edit the values below</b>, then run this cell (Shift+Enter or click &#x25B6; Run in the toolbar).</div>


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────
ngram_size       <- 1      # 1 = unigrams, 2 = bigrams, 3 = trigrams
remove_stopwords <- FALSE  # TRUE = exclude common function words
stopword_language <- "en"  # language for stopwords (e.g. "en", "de", "fr")
remove_punctuation <- TRUE # TRUE = exclude punctuation tokens
to_lowercase     <- TRUE   # TRUE = treat "The" and "the" as the same word
top_n            <- 30     # number of top items to show in the plot
min_freq         <- 2      # minimum frequency to include in output


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 3</span><br><b style="font-size:.98rem;">&#x1F4CA; Run the analysis</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
suppressPackageStartupMessages(library(quanteda))
suppressPackageStartupMessages(library(quanteda.textstats))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(dplyr))

seed_demo_texts("MyTexts")

texts <- tryCatch(load_texts("MyTexts"),
  error = function(e) { .err(conditionMessage(e)); NULL })

if (!is.null(texts)) {
  show_corpus_summary(texts)
  .prog("Tokenising...", 20)
  toks <- tokens(corpus(texts),
                 remove_punct = remove_punctuation,
                 remove_symbols = TRUE,
                 remove_numbers = FALSE)
  if (to_lowercase) toks <- tokens_tolower(toks)
  if (remove_stopwords)
    toks <- tokens_remove(toks,
      tryCatch(stopwords(stopword_language),
               error=function(e) { .warn(paste("Unknown stopword language:",
                 stopword_language, "-- using English.")); stopwords("en") }),
      padding=FALSE)
  if (ngram_size > 1)
    toks <- tokens_ngrams(toks, n=ngram_size)
  .prog("Building frequency list...", 50)
  dfm_obj <- dfm(toks)
  freq_df <- textstat_frequency(dfm_obj) |>
    as.data.frame() |>
    dplyr::filter(frequency >= min_freq) |>
    dplyr::select(feature, frequency, rank, docfreq) |>
    dplyr::mutate(
      norm_per_1000 = round(frequency / sum(frequency) * 1000, 4)
    )
  names(freq_df) <- c("Token","Frequency","Rank","DocFrequency","Freq_per_1000")
  # Corpus-level statistics
  total_tokens <- sum(freq_df$Frequency)
  total_types  <- nrow(freq_df)
  ttr          <- round(total_types / total_tokens, 4)
  hapax        <- sum(freq_df$Frequency == 1)
  IRdisplay::display_html(paste0(
    '<div style="font-family:sans-serif;margin:.5rem 0;">',
    '<b style="color:#51247a;font-size:.88rem;">Corpus statistics</b>',
    '<table style="border-collapse:collapse;margin-top:6px;">',
    '<tr><td style="padding:3px 14px;font-size:.82rem;">Total tokens</td>',
    '<td style="padding:3px 14px;font-size:.82rem;font-weight:700;">',
    format(total_tokens, big.mark=","), '</td></tr>',
    '<tr style="background:#f4f0f8;"><td style="padding:3px 14px;font-size:.82rem;">Unique types</td>',
    '<td style="padding:3px 14px;font-size:.82rem;font-weight:700;">',
    format(total_types, big.mark=","), '</td></tr>',
    '<tr><td style="padding:3px 14px;font-size:.82rem;">Type-token ratio (TTR)</td>',
    '<td style="padding:3px 14px;font-size:.82rem;font-weight:700;">', ttr, '</td></tr>',
    '<tr style="background:#f4f0f8;"><td style="padding:3px 14px;font-size:.82rem;">Hapax legomena</td>',
    '<td style="padding:3px 14px;font-size:.82rem;font-weight:700;">',
    format(hapax, big.mark=","), '</td></tr>',
    '</table></div>'
  ))
  .prog("Plotting top frequencies...", 70)
  item_label <- switch(as.character(ngram_size),
    "1"="Word", "2"="Bigram", "3"="Trigram", "N-gram")
  p_freq <- ggplot(head(freq_df, top_n),
                   aes(x=reorder(Token, Frequency), y=Frequency)) +
    geom_col(fill="#51247a", width=.7) +
    geom_text(aes(label=Frequency), hjust=-0.1, size=3, colour="gray30") +
    coord_flip() +
    scale_y_continuous(expand=expansion(mult=c(0,.15))) +
    theme_bw(base_size=12) +
    labs(title=paste0("Top ", top_n, " ", tolower(item_label), "s by frequency"),
         x=item_label, y="Frequency")
  print(p_freq)
  .prog("Plotting Zipf distribution...", 85)
  p_zipf <- ggplot(freq_df, aes(x=log10(Rank), y=log10(Frequency))) +
    geom_point(colour="#51247a", alpha=0.4, size=1.2) +
    geom_smooth(method="lm", se=TRUE, colour="#f0a500",
                linewidth=1, formula=y~x) +
    theme_bw(base_size=12) +
    labs(title="Zipf distribution (log rank vs log frequency)",
         x="log₁₀(Rank)", y="log₁₀(Frequency)")
  print(p_zipf)
  save_excel(freq_df, "frequency_list.xlsx")
  dir.create("MyOutput", showWarnings=FALSE, recursive=TRUE)
  ggsave("MyOutput/frequency_plot.png",  plot=p_freq,
         bg="white", width=8, height=6, dpi=200)
  ggsave("MyOutput/zipf_plot.png", plot=p_zipf,
         bg="white", width=7, height=5, dpi=200)
  .ok("Saved frequency_list.xlsx, frequency_plot.png, zipf_plot.png to MyOutput.")
}


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 4</span><br><b style="font-size:.98rem;">&#x2B07;&#xFE0F; Download your results</b></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:12px 18px;margin:.6rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2B07;&#xFE0F; Download your results</b><br>Files saved: <b>frequency_list.xlsx, frequency_plot.png, zipf_plot.png</b><br>Open the <b>file browser</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> any file and choose <b>Download</b>.</div>


---

### Citation

Schweinberger, Martin. (2025). *LADAL Frequency Analyser*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025frequency,
  author = {Schweinberger, Martin},
  title  = {LADAL Frequency Analyser},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
# Uncomment to show package versions for reproducibility
# sessionInfo()
